# ⚡ EC1. Optimización de Costes en "Fly-Next"

**Material desarrollado por:** Javier Morales, equipo [IA4LEGOS](https://ia4legos.umh.es/)

**Tema:** Generación de variables aleatorias.

**Licencia:** [CC BY-SA 4.0](http://creativecommons.org/licenses/by-sa/4.0/)

No olvides hacer una copia de este cuaderno (`Archivo > Guardar una copia en Drive`) antes de empezar a trabajar.

In [ ]:
#%%capture
# @title ⚠️ Cargar configuración del cuaderno
# Cargamos módulos de análisis numérico
import numpy as np          # importamos numpy como np
import pandas as pd         # importamos pandas como pd
import math                 # importamos módulo para cáculos matemáticos
import random
import inspect
from scipy import stats
import matplotlib.colors as mcolors # Importamos matplotlib.colors

# Cargamos módulos de análisis gráficos
from plotnine import *      # importamos módulo para gráficos con ggplot
# %matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
# %config InlineBackend.figure_format = 'retina'

#===============================================
# Cargar funciones bloque1
from urllib.request import urlretrieve
import re # for text manipulation


#url = 'https://raw.githubusercontent.com/asunmayoral/umh1477/refs/heads/main/bloque1_sps.py'
url = 'https://raw.githubusercontent.com/UMH1477/python/refs/heads/main/bloque1_sps.py'
urlretrieve(url, "bloque1_sps.py")
import bloque1_sps
from bloque1_sps import *

# visualización de las funciones precargadas
functions = []
for name, obj in inspect.getmembers(bloque1_sps):
    if inspect.isfunction(obj) and obj.__module__ == bloque1_sps.__name__:
        functions.append(name)

print("\nFunciones precargadas en bloque1_sps.py:\n")
for func_name in functions:
    print(f"- {func_name}")

# 1. Introducción

La empresa de logística **"Fly-Next"** quiere modelar el coste diario de sus operaciones de entrega de paquetes. El proceso de asignación de rutas y el cálculo de los costes asociados es estocástico, ya que depende de la naturaleza del destino, las condiciones del tráfico y la eficiencia del transporte utilizado en cada entrega.

Como Científico de Datos del equipo de operaciones, tu misión es simular el escenario de reparto para determinar el **coste esperado** y el **riesgo financiero** asociado a la operación diaria.

Este estudio de caso está organizado en dos bloques:

* **Bloque Básico** modela el coste de una entrega individual en función del tipo de ruta asignada, sin tener en cuenta el momento del día en que se produce.
* **Bloque Avanzado** amplía ese modelo incorporando **tres franjas horarias con distinta intensidad de tráfico**, que modifican tanto el volumen de pedidos que llegan al sistema como el retraso y el consumo de cada entrega.

# <font color="brown">**1. Tu encargo**</font>

No se te pide solo código: se te pide una **recomendación fundamentada**. Al terminar cada bloque deberás entregar dos productos, como haría cualquier asesor de ciencia de datos en un proyecto real de consultoría:

* Un **informe técnico** que recoja el planteamiento del problema, el modelo utilizado, los resultados de la simulación (con sus intervalos de confianza — nunca un único número suelto) y tu interpretación de negocio de cada resultado.
* Una **presentación ejecutiva** pensada para el comité de dirección de "Fly-Next": personas que no van a leer tu código ni tus fórmulas, y que necesitan entender, en el menor tiempo posible, cuál es el coste y el riesgo de la operación analizada, y qué recomiendas hacer al respecto.

Al final de cada bloque encontrarás el **encargo concreto de la Dirección**, redactado como una petición formal y organizado por objetivos — esa misma organización por objetivos es la que te conviene usar como esqueleto de tu informe técnico.

# 🅰️ <font color="brown">**2. Coste de una entrega**</font>

A continuación se describe la situación actual

## **2.1. El modelo de una entrega**

Cada paquete que entra en el sistema se asigna a una de tres posibles rutas mediante una variable indicadora discreta $T \in \{1, 2, 3\}$. La probabilidad de asignación de cada ruta es:

| Tipo de Ruta ($T$) | Descripción | Probabilidad ($p_j$) |
| :---: | :--- | :---: |
| **$T=1$** | Urbana (entregas locales) | $0.60$ |
| **$T=2$** | Regional (provincias cercanas) | $0.30$ |
| **$T=3$** | Premium (urgente / larga distancia) | $0.10$ |

Una vez asignada la ruta, el comportamiento de la entrega queda determinado por tres variables aleatorias **independientes entre sí, condicionadas al valor de $T$**: la **Distancia ($D$)**, el **Consumo ($G$)** y el **Retraso ($M$)**. Sus distribuciones varían según la ruta asignada:

| Ruta ($T$) | Distancia $D\mid T$ (km)__________ | Consumo $G\mid T$ (L/km)__________ | Retraso $M\mid T$ (horas)__________ |
| :--- | :---: | :---: | :---: |
| **1. Urbana** | $Po(\lambda=15)$ | $U(0.10,\ 0.15)$ | $Exp(\text{media}=0.5)$ |
| **2. Regional** | $N(\mu=80,\ \sigma=10)$ | $U(0.08,\ 0.12)$ | $Exp(\text{media}=1.2)$ |
| **3. Premium** | $N(\mu=300,\ \sigma=50)$ | $U(0.05,\ 0.07)$ | $Exp(\text{media}=3.0)$ |

## **2.2. Función de coste**

El coste total de una entrega individual ($Y$) es una transformación de las tres variables anteriores mediante la función $h(D, G, M)$:

$$Y = (D \cdot G \cdot P_{fuel}) + (M \cdot C_{labor}) + \mathbb{I}_{\{D > 100\}} \cdot K$$

| Parámetro | Valor | Significado |
| :--- | :---: | :--- |
| $P_{fuel}$ | $1.5$ €/litro | Precio del combustible |
| $C_{labor}$ | $25$ €/hora | Coste de mano de obra por retraso |
| $K$ | $50$ € | Recargo fijo por peajes si $D > 100$ km |

## **2.3. Algoritmo de simulación**

$T$, $D$, $G$ y $M$ se simulan mediante el **método de composición**: primero se simula la ruta y, después, se simula de la distribución condicional correspondiente.

1. Simular $t_i \sim T$ (rutas 1 a 3, con probabilidades $0.60,\ 0.30,\ 0.10$).
2. Simular, condicionado a $t_i$: $d_i, g_i, m_i$ según la Tabla de distribuciones condicionadas.
3. Calcular el coste $y_i = d_i g_i \cdot 1.5 + m_i \cdot 25 + 50\cdot \mathbb{I}_{\{d_i>100\}}$.
4. Repetir los pasos 1-3 $nsim$ veces para obtener la muestra $\{y_i\}$.

In [ ]:
# @title **Parámetros del modelo**

# --- 1.1 Distribución del indicador de ruta T ---
RUTAS = [1, 2, 3]                      # 1=Urbana, 2=Regional, 3=Premium
PROBS_RUTA = [0.60, 0.30, 0.10]

# --- 1.2 Distribuciones condicionadas D|T, G|T, M|T ---
# Cada entrada usa el nombre de la distribución en scipy.stats y sus
# argumentos de forma/localización/escala, tal y como se usarían en
# stats.<dist>.rvs(size=..., **kwargs)
PARAMS_RUTA = {
    1: dict(  # Urbana
        dist_D=("poisson", dict(mu=15)),
        dist_G=("uniform", dict(loc=0.10, scale=0.15 - 0.10)),
        dist_M=("expon",   dict(scale=0.5)),
    ),
    2: dict(  # Regional
        dist_D=("norm",    dict(loc=80, scale=10)),
        dist_G=("uniform", dict(loc=0.08, scale=0.12 - 0.08)),
        dist_M=("expon",   dict(scale=1.2)),
    ),
    3: dict(  # Premium
        dist_D=("norm",    dict(loc=300, scale=50)),
        dist_G=("uniform", dict(loc=0.05, scale=0.07 - 0.05)),
        dist_M=("expon",   dict(scale=3.0)),
    ),
}

# --- 1.3 Parámetros económicos de la función de coste Y = h(D, G, M) ---
P_FUEL = 1.5      # €/litro -- precio del combustible
C_LABOR = 25.0    # €/hora  -- coste de mano de obra por retraso
K_PEAJE = 50.0    # €       -- recargo fijo por peajes si D > UMBRAL_D
UMBRAL_D = 100.0  # km      -- distancia a partir de la cual se aplica el peaje

In [ ]:
# @title **Generador de pedidos**

def _generar_pedidos(n):
    """
    Genera las variables base (T, D, G, M, Peaje) de `n` pedidos mediante
    el método de composición: primero se simula la ruta T y después,
    condicionado a ella, las variables D, G y M.

    Esta función es el "núcleo" reutilizado tanto por el simulador del
    Bloque A (simulador) como por el del Bloque B (simulador_periodo),
    de modo que ambos comparten exactamente la misma lógica de
    generación de pedidos.

    Returns
    -------
    T, D, G, M, Peaje : arrays de numpy, todos de longitud n
    """
    T = np.random.choice(RUTAS, size=n, p=PROBS_RUTA)
    D = np.zeros(n)
    G = np.zeros(n)
    M = np.zeros(n)

    for ruta, cfg in PARAMS_RUTA.items():
        mask = (T == ruta)
        n_ruta = int(mask.sum())
        if n_ruta == 0:
            continue

        dist_name, kwargs = cfg["dist_D"]
        D[mask] = getattr(stats, dist_name).rvs(size=n_ruta, **kwargs)

        dist_name, kwargs = cfg["dist_G"]
        G[mask] = getattr(stats, dist_name).rvs(size=n_ruta, **kwargs)

        dist_name, kwargs = cfg["dist_M"]
        M[mask] = getattr(stats, dist_name).rvs(size=n_ruta, **kwargs)

    peaje = (D > UMBRAL_D).astype(int)
    return T, D, G, M, peaje

In [ ]:
# @title **Simulador**

def simulador(nsim):
    """
    Genera un banco de datos con `nsim` entregas simuladas de "Fly-Next",
    sin tener en cuenta franjas horarias ni tráfico (modelo base).

    Returns
    -------
    pd.DataFrame con columnas ['T', 'D', 'G', 'M', 'Peaje', 'Y']
    """
    T, D, G, M, peaje = _generar_pedidos(nsim)
    Y = D * G * P_FUEL + M * C_LABOR + K_PEAJE * peaje

    return pd.DataFrame({"T": T, "D": D, "G": G, "M": M, "Peaje": peaje, "Y": Y})


Comprobamos el simulador para conseguir $nsim=10.000$ entregas simuladas:

In [ ]:
NSIM = 10_000
datos_A = simulador(NSIM)
datos_A.head(10)

## **2.4. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las 10000 simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo  a un conjunto de datos. Para el análsiis de escenario climáticos basta con describir los resulttdos de dicha variable.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por la empresa y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por tipo de ruta, son coherentes con los parámetros teóricos.

## **2.5 El encargo de la Dirección**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección de Operaciones, "Fly-Next"
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Diagnóstico del coste y el riesgo financiero de las entregas
>
> Antes de decidir cambios en nuestra operación de reparto, necesitamos entender con números —no con impresiones— el coste real y el riesgo financiero de cada entrega. Trabajad con la muestra de 10.000 entregas que habéis generado (`datos_A`) y acompañad **cada estimación de su intervalo de confianza al 95%**: una cifra sin margen de error no nos sirve para tomar una decisión de este calado. Os hemos organizado la petición en tres objetivos.

### Objetivo 1. Cuantificar el riesgo económico de una entrega

Necesitamos saber, en cifras concretas, qué podemos esperar de una entrega cualquiera y hasta qué punto se pueden torcer las cosas en el peor de los casos.

* **O1.1.** Estimad el **coste esperado** $E(Y)$ y la **probabilidad** $Pr(Y>150)$.
* **O1.2.** Calculad el **percentil 90** de $Y$, mediante macro-réplicas. ¿Qué significado tiene este valor para la planificación operativa, frente a $E(Y)$?
* **O1.3.** Repetid el cálculo de $E(Y)$ y $Pr(Y>150)$ condicionando por tipo de ruta, y estimad además el **coeficiente de variación** de $Y$ para cada ruta por separado. ¿Qué ruta contribuye más a la cola derecha de la distribución global, y cuál presenta mayor incertidumbre relativa?

### Objetivo 2. Entender qué hay detrás de esa variabilidad

Antes de proponer ninguna solución, queremos saber qué parte de nuestra operación explica realmente el riesgo.

* **O2.1.** Descomponed, por simulación, la varianza total de $Y$ en la parte atribuible a cada uno de los tres términos de $Y=D\cdot G\cdot P_{fuel}+M\cdot C_{labor}+K\cdot\mathbb{I}_{\{D>100\}}$. ¿Qué término domina la variabilidad del coste?
* **O2.2.** Diseñad un análisis de sensibilidad que identifique cuál de los tres parámetros económicos ($P_{fuel}$, $C_{labor}$, $K$) tiene mayor impacto en la varianza de $Y$ (variando cada parámetro y manteniendo los demás fijos): necesitamos saber dónde concentrar nuestros esfuerzos.

### Objetivo 3. Explorar palancas de mejora de la operación actual

Antes de cambiar de estrategia de raíz, queremos agotar los ajustes más baratos sobre la operación que ya tenemos.

* **O3.1.** Si quisiéramos reducir el coste esperado un 10% cambiando únicamente el vector de probabilidades $(p_1,p_2,p_3)$ de las rutas, sin tocar las distribuciones condicionadas, planteadnos formalmente el problema de optimización correspondiente (función objetivo y restricciones) y resolvedlo por simulación.
* **O3.2.** Reescribid la función de coste $h(D,G,M)$ si, en lugar de un recargo fijo $K$, el peaje fuese proporcional a la distancia por encima de 100 km (por ejemplo, $0.15$€/km adicional). Comparad, por simulación, la distribución de $Y$ resultante con la del modelo actual.
* **O3.3.** Estimad, mediante simulación, el ahorro anual esperado si negociamos una reducción del 20% en el parámetro $K$ (recargo de peajes), asumiendo 50.000 entregas al año. Enumerad los supuestos necesarios.
* **O3.4.** Planteadnos cómo evaluaríais, mediante simulación tipo "A/B testing", si cambiar de proveedor de combustible (que reduciría $P_{fuel}$ de $1.5$ a $1.35$ €/L pero aumentaría el consumo medio $G$ un 5% por menor calidad) es una decisión rentable para la empresa.

# 🅱️ 3. Franjas horarias e intensidad de tráfico

El modelo del Bloque A trata todas las entregas como intercambiables en el tiempo. En la operación real de "Fly-Next" esto no es así: el volumen de pedidos y las condiciones de circulación cambian a lo largo del día, y ambas cosas afectan al coste. Vamos a incorporar esta dimensión temporal definiendo tres franjas horarias con distinta intensidad de tráfico.

## 3.1. El modelo temporal y el efecto del tráfico sobre el coste

El día se divide en 24 bloques horarios, cada uno asignado a una franja. El número de pedidos que entran al sistema durante cada hora se modela como un **proceso de Poisson cuya tasa depende de la franja**:

$$N_h \sim Poisson(\lambda_{f(h)}), \qquad h = 0, 1, \dots, 23$$

**Tabla 3. Franjas horarias, intensidad de tráfico y tasa de llegada de pedidos**

| Franja | Horario | Nº horas/día | Intensidad de tráfico | $\lambda$ (pedidos/hora) |
| :--- | :--- | :---: | :---: | :---: |
| **Valle** | 00:00–07:00 y 22:00–24:00 | 9 | Baja | $8$ |
| **Llano** | 07:00–09:00, 12:00–17:00 y 20:00–22:00 | 9 | Media | $25$ |
| **Punta** | 09:00–12:00 y 17:00–20:00 | 6 | Alta | $45$ |

El tráfico no afecta a la distancia $D$, pero ralentiza la circulación (aumentando el retraso $M$) e incrementa el consumo por kilómetro (aumentando $G$). Este efecto depende también del tipo de ruta: una entrega **Urbana** circula íntegramente por vías congestionadas, mientras que una **Premium** (autovía) apenas lo nota.

**Tabla 4. Factor multiplicativo sobre el retraso $M$**

| Franja | Urbana ($T=1$) | Regional ($T=2$) | Premium ($T=3$) |
| :--- | :---: | :---: | :---: |
| Valle | $1.00$ | $1.00$ | $1.00$ |
| Llano | $1.40$ | $1.20$ | $1.05$ |
| Punta | $2.20$ | $1.60$ | $1.10$ |

**Tabla 5. Factor multiplicativo sobre el consumo $G$**

| Franja | Urbana ($T=1$) | Regional ($T=2$) | Premium ($T=3$) |
| :--- | :---: | :---: | :---: |
| Valle | $1.00$ | $1.00$ | $1.00$ |
| Llano | $1.08$ | $1.04$ | $1.00$ |
| Punta | $1.20$ | $1.10$ | $1.02$ |

Para una entrega $i$ en la hora $h$ (franja $f(h)$) con ruta $T_i$, se calculan **dos costes** a partir de las MISMAS variables base $D_i,G_i,M_i$:

$$Y_i^{\,sin} = D_i G_i P_{fuel} + M_i C_{labor} + K\, \mathbb{I}_{\{D_i>100\}} \qquad\qquad Y_i^{\,con} = D_i G_i^{ajust} P_{fuel} + M_i^{ajust} C_{labor} + K\, \mathbb{I}_{\{D_i>100\}}$$

donde:
* $G_i^{ajust}=G_i\cdot\text{factor\_consumo}(f(h),T_i)$ y
* $M_i^{ajust}=M_i\cdot\text{factor\_retraso}(f(h),T_i)$.


El **sobrecoste atribuible al tráfico** es $\Delta_i = Y_i^{con}-Y_i^{sin}$.

> 💡 **Nota metodológica — simulación pareada.** $Y_i^{sin}$ e $Y_i^{con}$ comparten exactamente las mismas variables base: solo cambia el multiplicador determinista que se les aplica. Esta técnica de **números aleatorios comunes** produce una estimación de $\Delta_i$ mucho más precisa que si se simularan ambos escenarios de forma independiente.

## 3.2. Algoritmo de simulación extendido

1. Para cada hora $h$ del día, determinar su franja $f(h)$ (Tabla 3).
2. Simular $n_h\sim Poisson(\lambda_{f(h)})$ pedidos para esa hora.
3. Simular $T_i, D_i, G_i, M_i$ para cada pedido, condicionados a la ruta (igual que en el Bloque A).
4. Aplicar los factores de tráfico (Tablas 4 y 5) y calcular $Y_i^{sin}$ e $Y_i^{con}$ a partir de las mismas $D_i,G_i,M_i$.
5. Calcular el sobrecoste $\Delta_i = Y_i^{con}-Y_i^{sin}$.
6. Repetir los pasos 1-5 para las 24 horas del día y para los $ndias$ días del periodo.

In [ ]:
# @title **Parámetros del modelo temporal**

# --- 6.1 Asignación de cada hora del día (0-23) a una franja horaria ---
HORAS_FRANJA = {}
for h in list(range(0, 7)) + [22, 23]:
    HORAS_FRANJA[h] = "Valle"
for h in [7, 8] + list(range(12, 17)) + [20, 21]:
    HORAS_FRANJA[h] = "Llano"
for h in list(range(9, 12)) + list(range(17, 20)):
    HORAS_FRANJA[h] = "Punta"

FRANJAS = ["Valle", "Llano", "Punta"]

# --- 6.2 Tasa de llegada de pedidos por hora, según la franja (Tabla 3) ---
LAMBDA_FRANJA = {"Valle": 8, "Llano": 25, "Punta": 45}

# --- 6.3 Factores multiplicativos de tráfico sobre M y G (Tablas 4 y 5) ---
FACTOR_RETRASO = {
    "Valle": {1: 1.00, 2: 1.00, 3: 1.00},
    "Llano": {1: 1.40, 2: 1.20, 3: 1.05},
    "Punta": {1: 2.20, 2: 1.60, 3: 1.10},
}
FACTOR_CONSUMO = {
    "Valle": {1: 1.00, 2: 1.00, 3: 1.00},
    "Llano": {1: 1.08, 2: 1.04, 3: 1.00},
    "Punta": {1: 1.20, 2: 1.10, 3: 1.02},
}

In [ ]:
# @title **Generador de datos: simulador_periodo(ndias)**

def simulador_periodo(ndias):
    """
    Simula `ndias` días completos de operación de "Fly-Next", teniendo en
    cuenta las tres franjas horarias, sus tasas de llegada de pedidos y
    el efecto multiplicativo del tráfico sobre el retraso (M) y el
    consumo (G) de cada entrega.

    Algoritmo (ver sección 3.2): para cada hora se determina la franja y
    se simula el nº de pedidos Poisson(lambda_franja); las variables base
    de todos los pedidos del periodo se generan de una sola vez y después
    se les aplican los factores de tráfico según (franja, ruta) de cada
    uno. La función está vectorizada (sin bucles por pedido ni por hora
    en Python puro) para poder generar muchos días y réplicas rápido.

    Returns
    -------
    pd.DataFrame con un registro por pedido y columnas:
        ['Dia', 'Hora', 'Franja', 'T', 'D', 'G', 'M', 'Peaje',
         'G_ajustado', 'M_ajustado', 'Y_sin_trafico', 'Y_con_trafico',
         'Sobrecoste_trafico']
    """
    horas = np.arange(24)
    franja_por_hora = np.array([HORAS_FRANJA[h] for h in horas])
    lam_por_hora = np.array([LAMBDA_FRANJA[f] for f in franja_por_hora])

    # Repetimos la estructura horaria para los `ndias` días
    dias_arr = np.repeat(np.arange(1, ndias + 1), 24)
    horas_arr = np.tile(horas, ndias)
    franja_arr = np.tile(franja_por_hora, ndias)
    lam_arr = np.tile(lam_por_hora, ndias)

    # 2. Nº de pedidos que llegan en cada bloque día-hora
    n_pedidos_bloque = np.random.poisson(lam_arr)
    total_n = int(n_pedidos_bloque.sum())

    # "Desplegamos" cada bloque horario tantas veces como pedidos genera
    Dia = np.repeat(dias_arr, n_pedidos_bloque)
    Hora = np.repeat(horas_arr, n_pedidos_bloque)
    Franja = np.repeat(franja_arr, n_pedidos_bloque)

    # 3. Variables base de todos los pedidos del periodo, de una vez
    T, D, G, M, Peaje = _generar_pedidos(total_n)

    # 4. Factores de tráfico según (franja, ruta) de cada pedido
    f_ret = np.ones(total_n)
    f_con = np.ones(total_n)
    for franja in FRANJAS:
        for ruta in RUTAS:
            mask = (Franja == franja) & (T == ruta)
            f_ret[mask] = FACTOR_RETRASO[franja][ruta]
            f_con[mask] = FACTOR_CONSUMO[franja][ruta]

    G_ajustado = G * f_con
    M_ajustado = M * f_ret

    # 5. Coste con y sin tráfico (simulación pareada: mismas D, G, M base)
    Y_sin = D * G * P_FUEL + M * C_LABOR + K_PEAJE * Peaje
    Y_con = D * G_ajustado * P_FUEL + M_ajustado * C_LABOR + K_PEAJE * Peaje

    datos = pd.DataFrame({
        "Dia": Dia, "Hora": Hora, "Franja": Franja, "T": T,
        "D": D, "G": G, "M": M, "Peaje": Peaje,
        "G_ajustado": G_ajustado, "M_ajustado": M_ajustado,
        "Y_sin_trafico": Y_sin, "Y_con_trafico": Y_con,
        "Sobrecoste_trafico": Y_con - Y_sin,
    })
    return datos

Podemos simular un periodo completo de operación con:

In [ ]:
N_DIAS = 30
datos_B = simulador_periodo(N_DIAS)
datos_B.head(10)

## **3.3. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo a un conjunto de datos. Para el análisis del número de pedidos por hora basta con describir los resultados de dicha variable.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por la empresa y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por franja horaria, son coherentes con los parámetros teóricos.

Obtén, con su IC al 95%, mediante macro-réplicas de periodos completos:

* El **coste diario esperado**, con y sin tráfico.
* El **sobrecoste diario esperado** por tráfico (y qué % supone).
* La **contribución de cada franja horaria** al sobrecoste total.
* El **percentil 95** del coste diario total.

## **3.4 El encargo de la Dirección (fase 2): ¿cómo gestionamos el impacto del tráfico?**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección de Operaciones, "Fly-Next"
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Evaluación del impacto del tráfico y la congestión horaria sobre el coste de reparto
>
> Gracias por el diagnóstico de la fase anterior. Ahora necesitamos que evaluéis cuánto nos cuesta la congestión de las franjas de mayor tráfico y qué podemos hacer al respecto. Trabajad siempre sobre los registros pareados de `datos_B` (`Y_sin_trafico` / `Y_con_trafico`), generados con números aleatorios comunes, y acompañad cada estimación de su intervalo de confianza al 95%. Os hemos organizado la petición en cuatro objetivos, más una ampliación optativa.

### Objetivo 4. Comparar, pedido a pedido, el coste con y sin tráfico

* **O4.1.** Estimad, para cada franja horaria, el **coste medio por pedido** con y sin tráfico, $E(Y_{con})$ y $E(Y_{sin})$, y el **sobrecoste medio** $E(\text{Sobrecoste\_trafico})$. Aprovechad que el diseño es pareado (números aleatorios comunes) para explicarnos por qué el intervalo de confianza del sobrecoste sale mucho más estrecho que si hubierais simulado ambos escenarios de forma independiente.
* **O4.2.** A nivel diario, estimad el **coste total esperado** $E(\text{Coste\_diario})$ y el **sobrecoste diario esperado** $E(\text{Sobrecoste\_diario})$, y expresad este último como porcentaje del coste diario total.
* **O4.3.** Calculad el **percentil 95** del coste diario total mediante macro-réplicas, y complementadlo con el **CVaR al 5%** (el coste medio en el 5% de los días más caros). ¿Qué información adicional aporta el CVaR frente al percentil 95?

### Objetivo 5. Saber de qué depende el sobrecoste por tráfico

* **O5.1.** La empresa se plantea desplazar el 20% de los pedidos Urbanos ($T=1$) de la franja Punta hacia la franja Valle. Modificad el simulador para reflejar este escenario y estimad el ahorro esperado en sobrecoste diario. ¿Compensa el ahorro los previsibles costes operativos y comerciales del cambio?
* **O5.2.** "Fly-Next" contempla externalizar las entregas Premium a un proveedor especializado por un coste fijo de 120€/entrega, en lugar de simularlas con el modelo actual. Diseñad la comparación de escenarios (coste esperado y riesgo) que presentaríais a dirección para decidir si compensa la externalización.
* **O5.3.** Calculad la contribución relativa de cada franja al sobrecoste total diario (como fracción del sobrecoste total) y determinad, aproximadamente, qué reducción porcentual del factor de retraso en Punta haría que Llano y Punta contribuyeran por igual.

### Objetivo 6. Dimensionar la operación y comprobar su robustez

* **O6.1.** Plantead cómo estimaríais, por simulación, la **flota mínima** de vehículos/conductores necesaria para atender, con al menos un 95% de probabilidad, toda la demanda de la franja Punta sin retrasos adicionales por falta de capacidad (más allá del efecto de tráfico ya modelado).
* **O6.2.** Plantead cómo adaptaríais el procedimiento de macro-réplicas para estimar, en lugar del percentil 95, la **probabilidad de que el coste diario supere un presupuesto fijo de 30.000€**. Dad el estimador y su intervalo de confianza.
* **O6.3.** Proponed cómo modificaríais el simulador para incorporar un **límite máximo de vehículos disponibles por hora** (censurando el número de pedidos atendidos), y qué nueva variable de interés podríais calcular (por ejemplo, "pedidos rechazados"). Comparad, conceptualmente, el escenario con límite de flota frente al escenario actual sin límite.

### Objetivo 7. Vuestra recomendación

* **O7.1.** A partir de los resultados de los Objetivos 4 a 6, redactadnos la recomendación sobre cómo gestionar el impacto del tráfico en la operación de reparto, y diseñad un **cuadro de mando conceptual** (no hace falta programarlo) con los KPIs mínimos que reportaríais mensualmente a la dirección de "Fly-Next", justificando la relevancia de cada uno.

## Objetivo 8. Ampliación optativa

Lo que sigue **no forma parte del encargo formal** de la Dirección: es un banco de cuestiones adicionales, de mayor dificultad, para quien quiera explorar el modelo de franjas horarias con más profundidad.

* **O8.1.** El número total de pedidos diarios es la suma de 24 variables de Poisson independientes con tasas distintas. ¿Qué distribución tiene esa suma? Dad su media y su varianza en función de las $\lambda$ de la Tabla 3.
* **O8.2.** Retomad el escenario de desplazar el 20% de los pedidos Urbanos de Punta a Valle (Objetivo 5). En lugar de un porcentaje fijo, modelad el desplazamiento como el resultado de un incentivo económico variable y plantead, por simulación, cómo encontraríais el porcentaje de desplazamiento que maximiza el beneficio neto de la empresa (ahorro en sobrecoste menos coste del incentivo).
* **O8.3.** El sobrecoste por tráfico representa aproximadamente una quinta parte del coste diario total. Diseñad una campaña de mitigación combinada (parte se desplaza de franja, parte se compensa con vehículos más eficientes) y estimad, por simulación y con un presupuesto que defináis vosotros, cuánto sobrecoste conseguiríais eliminar.
* **O8.4.** Demostrad, empíricamente, por qué el diseño pareado (mismas $D,G,M$ base para $Y^{sin}$ e $Y^{con}$) reduce la varianza de la estimación del sobrecoste por tráfico, frente a simular ambos escenarios de forma independiente.
* **O8.5.** Investigad qué es un **proceso de Poisson doblemente estocástico** (proceso de Cox) y plantead cómo mejoraríais el modelo si, además de la franja horaria, la propia tasa de llegada de pedidos $\lambda$ fuera incierta día a día (por ejemplo, por estacionalidad o eventos especiales).
* **O8.6.** Comparad el modelo actual (sin límite de vehículos) con un modelo de colas $M/M/c$ en el que los pedidos que llegan sin vehículo disponible esperan en cola en lugar de rechazarse. ¿Qué sesgo introduce la simplificación actual del modelo respecto a un escenario con flota limitada?